# Use of Force Data Cleaning

In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/use_of_force.csv")

In [3]:
df.dtypes

Incident Number              int64
Subject_Incident Number    float64
Sex                            str
Race                           str
Subject Role                   str
Subject Age                float64
Block Address                  str
City                           str
Use of Force Number          int64
Officer_Incident Number      int64
Officer Role                   str
Use of Force Level             str
Use of Force Type              str
OccurredDate                   str
Subject Arrested               str
Use of Force Reason            str
Subject Altered State          str
NonBerkeley Address           bool
dtype: object

## Standardize Column Names & ID Types

In [5]:
# rename columns
rename_map = {"Incident Number": "incident_number", "Subject_Incident Number": "subject_incident_number", 
              "Sex": "sex", "Race": "race", "Subject Role": "subject_role", "Subject Age": "subject_age",
              "Block Address": "block_address", "City": "city", "Use of Force Number": "use_of_force_number",
              "Officer_Incident Number": "officer_incident_number", "Officer Role": "officer_role", 
              "Use of Force Level": "use_of_force_level", "Use of Force Type": "use_of_force_type",
              "OccurredDate": "occurred_date", "Subject Arrested": "subject_arrested", 
              "Use of Force Reason": "use_of_force_reason", "Subject Altered State": "subject_altered_state",
              "NonBerkeley Address": "non_berkeley_address"}
df = df.rename(columns=rename_map)

# convert ID columns to nullable integers 
id_cols = ["incident_number", "subject_incident_number", "use_of_force_number", "officer_incident_number"]
for col in id_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

## Filter to Berkeley & Timeframe (2021-03-01 to 2026-04-30)

In [11]:
# filter city to Berkeley
df["city"] = df["city"].astype(str).str.strip().str.title()
df = df[df["city"] == "Berkeley"].copy()

# filter timeframe
dt_temp = pd.to_datetime(df["occurred_date"], utc=True, errors="coerce")
start_ts = pd.Timestamp("2021-03-01", tz="UTC")
end_ts = pd.Timestamp("2026-04-30 23:59:59", tz="UTC")

df = df[(dt_temp >= start_ts) & (dt_temp <= end_ts)].copy()
dt_temp = dt_temp[(dt_temp >= start_ts) & (dt_temp <= end_ts)]

# format occurred_date to match team spec & extract temporal helper columns
df["occurred_date"] = dt_temp.dt.strftime("%Y-%m-%d %H:%M:%S+00")

# verification check
print(f"Filtered row count: {len(df):,}")
print(f"Unique City: {df["city"].unique().tolist()}")
print(f"Date range: {df['occurred_date'].min()} to {df['occurred_date'].max()}")

Filtered row count: 7,530
Unique City: ['Berkeley']
Date range: 2021-03-01 00:00:00+00 to 2026-04-30 00:00:00+00
